# Agents SDK Pipeline — демо в Colab

Целевая среда: Google Colab с GPU T4.

Этот notebook использует стандартные средства Colab для загрузки и скачивания файлов. Он рассчитан на **кейсы Level 1**: один базовый `.docx` и один `.docx` с изменениями.

Поддерживаемые операции:
- `replace_point` — изложить пункт в новой редакции
- `repeal_point` — признать пункт утратившим силу
- `insert_point` — вставить новый пункт
- `append_section_item` — дополнить подпункт или элемент
- `replace_phrase_globally` — заменить или удалить фразу по документу
- `replace_appendix_block` — изложить в новой редакции структурно однозначный блок приложения

**Ограничения:**
- Хранилище Colab временное и очищается после перезапуска runtime.
- Наличие T4 зависит от выбранного runtime в Colab.
- Ollama и модель устанавливаются внутри текущего runtime и при первом запуске занимают около 5 минут.
- Semantic ranking по умолчанию отключён.
- Топология multi-base (один документ изменений → несколько базовых документов) в этом демо не поддерживается.

In [ ]:
# ── Ячейка 1: Установка зависимостей и клонирование репозитория ───────────
!pip install -q 'openai>=1.76.0' 'python-docx>=1.1.2' 'langgraph>=0.2.70' 'langchain-core>=0.3.0'

from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/antonovska/RedActa.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/agents_sdk_pipeline_production")

if not REPO_DIR.exists():
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {REPO_BRANCH}

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print(f"Репозиторий: {REPO_DIR}")
print(f"Python path: {sys.path[0]}")

In [ ]:
# ── Ячейка 2: Установка Ollama и загрузка demo-модели (~5 мин на cold start) ──
import subprocess
import time
import urllib.request

MODEL_NAME = "hf.co/ai-sage/GigaChat3.1-10B-A1.8B-GGUF:Q4_K_M"

!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.ai/install.sh | sh

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

def wait_for_ollama(timeout_seconds: int = 900) -> None:
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as resp:
                if resp.status == 200:
                    print("Ollama готов. Дождитесь загрузки модели. ")
                    return
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama не стал готов в течение {timeout_seconds}с: {last_error}")

wait_for_ollama()
!ollama pull {MODEL_NAME}
print("Модель готова.")

In [ ]:
# ── Ячейка 3: Загрузка базового документа и документа изменений ───────────
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

uploaded_paths = {"base": None, "amendment": None}
upload_dir = REPO_DIR / "colab_uploads"
upload_dir.mkdir(exist_ok=True)

def upload_one(label: str) -> None:
    print(f"Выберите файл {label} .docx в диалоге ниже…")
    uploaded = files.upload()
    if not uploaded:
        print(f"Файл для {label} не выбран.")
        return
    name = next(iter(uploaded.keys()))
    if not name.lower().endswith(".docx"):
        raise ValueError(f"Ожидался .docx для {label}, получен: {name}")
    target = upload_dir / name
    target.write_bytes(uploaded[name])
    uploaded_paths[label] = target
    print(f"{label}: {target}")

base_btn = widgets.Button(description="Загрузить base .docx", button_style="info")
amend_btn = widgets.Button(description="Загрузить amendment .docx", button_style="info")

base_btn.on_click(lambda _: upload_one("base"))
amend_btn.on_click(lambda _: upload_one("amendment"))

display(widgets.HBox([base_btn, amend_btn]))
print("Нажмите кнопки, чтобы загрузить каждый документ.")

In [ ]:
# ── Ячейка 4: Запуск пайплайна ──────────────────────────────────────────────
import json
from graph_pipeline.colab_runner import run_uploaded_pair

result_state = {}

def run_demo(_=None) -> None:
    missing = [k for k, v in uploaded_paths.items() if v is None]
    if missing:
        print(f"Сначала загрузите: {', '.join(missing)}")
        return
    print("Запуск пайплайна… запаситесь терпением. Выдыхаем на шаге 4.")
    result = run_uploaded_pair(
        base_docx=uploaded_paths["base"],
        amendment_docx=uploaded_paths["amendment"],
        workspace_root=REPO_DIR,
        models_config=REPO_DIR / "config" / "models.colab.json",
        case_id="colab",
    )
    result_state.clear()
    result_state.update(result)
    validation = result.get("validation", {})
    print("\nСводка валидации:")
    print(json.dumps(validation, ensure_ascii=False, indent=2))
    print(f"\nВыходной документ: {result.get('output_doc')}")
    status = "valid" if validation.get("is_valid") else "invalid"
    print(f"Статус: {status}")
    total_markers = sum(len(step.get("revision_markers", [])) for step in result.get("steps", []))
    print(f"Маркеры редакции: {total_markers} вставлено")

run_btn = widgets.Button(description="Запустить пайплайн", button_style="success")
run_btn.on_click(run_demo)
display(run_btn)

In [ ]:
# ── Ячейка 5: Скачивание результата ────────────────────────────────────────
from google.colab import files as colab_files
import shutil

def download_result(_=None) -> None:
    output_doc = result_state.get("output_doc")
    if not output_doc:
        print("Сначала запустите пайплайн (ячейка 4).")
        return
    colab_files.download(output_doc)

def download_artifacts(_=None) -> None:
    artifacts_dir = REPO_DIR / "artifacts" / "colab"
    if not artifacts_dir.exists():
        print("Каталог artifacts не найден. Сначала запустите пайплайн.")
        return
    zip_path = shutil.make_archive(str(artifacts_dir), "zip", artifacts_dir)
    colab_files.download(zip_path)

dl_btn = widgets.Button(description="Скачать результат .docx", button_style="warning")
zip_btn = widgets.Button(description="Скачать artifacts .zip")

dl_btn.on_click(download_result)
zip_btn.on_click(download_artifacts)

display(widgets.HBox([dl_btn, zip_btn]))

## Что выполнилось

Notebook:
1. Клонировал репозиторий и установил зависимости.
2. Установил Ollama и загрузил модель GigaChat 10B Q4.
3. Загрузил ваши документы и разложил их во временную case-папку Colab.
4. Запустил тот же graph pipeline, что и CLI (`graph_pipeline.colab_runner.run_uploaded_pair`).
5. Вставил маркеры редакции (в скобках — источник изменения) в выходной документ.
6. Сохранил сгенерированные файлы в `artifacts/colab/`.

**Диагностика:**
- *Не загружается модель* — проверьте, что в Colab выбран runtime с GPU (Runtime → Change runtime type → T4 GPU).
- *Таймаут health-check Ollama* — повторно запустите ячейку 2.
- *Нет диалога загрузки или скачивания* — перезапустите только нужную ячейку; хранилище Colab временное в рамках сессии.
- *Статус `invalid`* — проверьте напечатанную сводку валидации и архив artifacts с debug JSON.
- *Служебная таблица* — в google colab при сохранении текста и форматирования служебной таблицы слетают конфиги при сохранении. Необходимо доработать для демо в Colab.